# Initial Data Inspection of Fridays For Future Insta Postings
***

## Only Posts by FridaysForFuture Accounts


## 1) Load Data

In [ ]:
import torch
torch.cuda.empty_cache()
print(torch.__version__)
torch.cuda.is_available()

In [ ]:
import sys
from pathlib import Path
import sshfs

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import plot_style
import json
import re
from collections import Counter
import yaml 

import torch
from spacy.lang.de.stop_words import STOP_WORDS as DE_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from src.crisis_windows import CRISIS_WINDOWS, resolve_windows, filter_by_window

# data server connection
#with open("C:/Users/Elena Solar/.ssh/sshfs_config.yaml") as f:
#    config = yaml.safe_load(f)
    
with open("../../login/sshfs_config.yaml") as f:
    config = yaml.safe_load(f)

fs = sshfs.SSHFileSystem(
    host=config["sshfs"]["host"],
    username=config["sshfs"]["username"],
    client_keys=config["sshfs"]["client_keys"],
)


plot_style.set_style()

In [ ]:
# Read and combine files

df_FFF = pd.read_csv("../data/processed/FFF_german.csv", low_memory = False)
df_FFF["creation_time"] = pd.to_datetime(df_FFF["creation_time"], utc=True)
df_FFF

### Data Frame Columns

* `content_type`: type of post, i.e. ['videos', 'albums', 'photos', 'stories']
* `creation_time`: timestamp the post was published
* `hashtags`: list of hashtags used in the post
* `id`: unique identifier of the post
* `is_branded_content`: whether the post is flagged as branded [False/True]
* `lang`: detected language of the post text
* `match_type`: how the post matched the search query/criteria, ['["post_text"]', '["multimedia_text"]', nan]
* `mcl_url`: URL to the post in the Meta Content Library
* `modified_time`: timestamp the post was last modified
* `multimedia`: attached media objects (images/videos) and their metadata
* `post_owner.id`: unique identifier of the posting account
* `post_owner.name`: display name of the posting account
* `post_owner.type`: account type, i.e. ['creator', 'business', 'personal']
* `post_owner.username`: handle/username of the posting account
* `statistics.comment_count`: number of comments on the post
* `statistics.like_count`: number of likes on the post
* `statistics.views`: number of views
* `statistics.views_date_last_refreshed`: timestamp the views count was last updated
* `text`: post caption/body text

---
## 2) Descriptive Statistics

### Content Type

In [ ]:
content_type_counts = df_FFF["content_type"].value_counts()
content_type_pct = (content_type_counts / content_type_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_SINGLE)
sns.countplot(data=df_FFF, x="content_type", order=content_type_counts.index, ax=ax)

# annotate with count (share%) instead of just count
for patch, count, pct in zip(ax.patches, content_type_counts, content_type_pct):
    ax.annotate(f"{count} ({pct}%)",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom", fontsize=9, xytext=(0, 2),
                textcoords="offset points")

ax.set_title("Posts by Content Type")
ax.set_xlabel("Content Type")
ax.set_ylabel("Number of Posts")
#plot_style.savefig(fig, "FFF/content_type_counts")
plt.show()


### Posting Volume Over Time

In [ ]:
daily_by_type = (
    df_FFF.groupby([df_FFF["creation_time"].dt.floor("D"), "content_type"])
    .size()
    .unstack(fill_value=0)
    .rolling(30).mean()
)
daily_total = daily_by_type.sum(axis=1)

daily_by_type_long = daily_by_type.reset_index().melt(
    id_vars="creation_time", var_name="content_type", value_name="posts_per_day"
)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.lineplot(
    data=daily_by_type_long, x="creation_time", y="posts_per_day",
    hue="content_type", linewidth=1, alpha=0.85, ax=ax,
)
ax.plot(daily_total.index, daily_total, color="black", linewidth=1.8, label="All posts")

ax.set_title("Post Volume Over Time (30-day rolling average)")
#ax.set_xlim(2018, 2026)
ax.set_xlabel("Date")
ax.set_ylabel("Posts per Day")
ax.legend(title="Content Type", loc="upper right")
plot_style.savefig(fig, "FFF/post_volume_timeline")
plt.show()

### Account Type

In [ ]:
owner_type_counts = df_FFF["post_owner.type"].value_counts()

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_SINGLE)
sns.countplot(data=df_FFF, x="post_owner.type", order=owner_type_counts.index, ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Posts by Account Type")
ax.set_xlabel("Account Type")
ax.set_ylabel("Number of Posts")
#plot_style.savefig(fig, "FFF/account_type_counts")
plt.show()

### Engagement: Likes & Views

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=plot_style.FIGSIZE_WIDE)
seq_colors = sns.color_palette(plot_style.PALETTE_SEQUENTIAL, 6)

sns.histplot(df_FFF["statistics.like_count"].dropna(), bins=50, ax=axes[0], color=seq_colors[2])
axes[0].set_yscale("log")
axes[0].set_title("Distribution of Like Counts")
axes[0].set_xlabel("Likes")

sns.histplot(df_FFF["statistics.views"].dropna(), bins=50, ax=axes[1], color=seq_colors[4])
axes[1].set_yscale("log")
axes[1].set_title("Distribution of View Counts")
axes[1].set_xlabel("Views")

plt.tight_layout()
#plot_style.savefig(fig, "FFF/likes_views_distributions")
plt.show()

### Language

In [ ]:
lang_counts = df_FFF["lang"].value_counts().head(15)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.countplot(
    data=df_FFF[df_FFF["lang"].isin(lang_counts.index)], y="lang", order=lang_counts.index, ax=ax,
)
plot_style.annotate_bars(ax)
ax.set_title("Top 15 Languages by Post Count")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Language")
#plot_style.savefig(fig, "FFF/language_counts")
plt.show()

### Top Accounts

In [ ]:
df_FFF["post_owner.username"].value_counts()

### Posts per Account and Year

Corpus composition over time -- which chapter is doing the posting in which year. The
national account dominates 2019-2020 and nearly disappears after 2022, while Hamburg
(2023-24) and Berlin (2025) carry the late corpus. Any framing trend over time is therefore
also a change in *who* is speaking, which is why the regression models in
`06_crisis_window_regression.ipynb` control for account.

Austrian chapters and the thin 2026 slice (~50 posts, Jan-Jul) are already excluded upstream
(`00_data_inspection.ipynb`, scope cell); the analysis period is Dec 2018 - Dec 2025.

In [ ]:
posts_by_account_year = pd.crosstab(
    df_FFF["post_owner.username"], df_FFF["creation_time"].dt.year, margins=True, margins_name="total"
)
display(posts_by_account_year)

counts = posts_by_account_year.drop(index="total", columns="total")
shares = counts / counts.sum()  # each account's share of that year's posts
display((shares * 100).round(0).astype(int).rename_axis(columns="share of year's posts (%)"))

with plot_style.scaled_fonts(plot_style.FIGSIZE_WIDE):
    fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
    counts.T.plot(kind="bar", stacked=True, ax=ax, color=plot_style.PALETTE_LINES[:len(counts)], width=0.8)
    # label each segment with its share of that year's posts -- only where the segment is tall
    # enough to hold the text (>= 30 posts); the exact shares for the thin years are in the table
    for container, account in zip(ax.containers, counts.index):
        labels = [f"{s:.0%}" if n >= 30 else "" for s, n in zip(shares.loc[account], counts.loc[account])]
        ax.bar_label(container, labels=labels, label_type="center", color="white")
    ax.set_title("Posts per Year by Account (labels: share of the year's posts)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of Posts")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Account", loc="upper left", bbox_to_anchor=(1.02, 1))
    plot_style.savefig(fig, "FFF/posts_per_account_year")
    plt.show()

### Content Type per Year

What kind of post carries the caption, by year. Photos dominate the founding phase (63% of
2019 posts) while videos grow from 11% (2019) to 37% (2025). Since only captions are scored,
a rising video share means the caption carries a shrinking part of the message -- a limitation
of the caption-only design that gets worse over the period (paper, Data section). Labels are
suppressed for segments under 30 posts (the thin 2018 slice); the exact shares are in the
table.

In [ ]:
posts_by_type_year = pd.crosstab(
    df_FFF["content_type"], df_FFF["creation_time"].dt.year, margins=True, margins_name="total"
)
display(posts_by_type_year)

type_counts = posts_by_type_year.drop(index="total", columns="total")
type_counts = type_counts.loc[type_counts.sum(axis=1).sort_values(ascending=False).index]  # most common type at the bottom
type_shares = type_counts / type_counts.sum()  # each content type's share of that year's posts
display((type_shares * 100).round(0).astype(int).rename_axis(columns="share of year's posts (%)"))

figsize = (plot_style.FIGSIZE_WIDE[0], 3.8)  # a little taller than FIGSIZE_WIDE so the thin 2025 bar still holds its labels
with plot_style.scaled_fonts(figsize) as scale:
    fig, ax = plt.subplots(figsize=figsize)
    type_counts.T.plot(kind="bar", stacked=True, ax=ax, color=plot_style.PALETTE_LINES[:len(type_counts)], width=0.8)
    # same labelling rule as the account plot: share of the year's posts, only for segments >= 30 posts
    for container, content_type in zip(ax.containers, type_counts.index):
        labels = [f"{s:.0%}" if n >= 30 else "" for s, n in zip(type_shares.loc[content_type], type_counts.loc[content_type])]
        ax.bar_label(container, labels=labels, label_type="center", color="white",
                     fontsize=plot_style.FONT_SIZE_ANNOTATION * scale)
    ax.set_title("Posts per Year by Content Type (labels: share of the year's posts)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of Posts")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Content Type", loc="upper left", bbox_to_anchor=(1.02, 1))
    plot_style.savefig(fig, "FFF/content_type_per_year")
    plt.show()

### Hashtags

In [ ]:
hashtag_lists = df_FFF["hashtags"].dropna().apply(json.loads)
hashtag_counts = Counter(tag.lower() for tags in hashtag_lists for tag in tags)

top_hashtags = pd.Series(dict(hashtag_counts.most_common(30)))
total_tags = sum(hashtag_counts.values())
top_hashtags_pct = (top_hashtags / len(df_FFF) * 100).round(1)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(x=top_hashtags.values, y=top_hashtags.index, order=top_hashtags.index, ax=ax)

# annotate with count (share%) instead of just count
for patch, count, pct in zip(ax.patches, top_hashtags, top_hashtags_pct):
    ax.annotate(f"{count} ({pct}%)",
                (patch.get_width(), patch.get_y() + patch.get_height() / 2),
                ha="left", va="center", fontsize=9, xytext=(4, 0),
                textcoords="offset points")

ax.set_title("Top 30 Hashtags")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Hashtag")
#plot_style.savefig(fig, "FFF/top_hashtags")
plt.show()

## 3) Quality Checks

### Substantive Text Content

Hashtags and (redacted) mentions inflate raw text length without carrying diagnostic/prognostic/motivational content. Strip both, then check how many posts still have enough words left to plausibly make a statement or mobilize people.

In [ ]:
def strip_hashtags_mentions(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"#\S+", " ", text)              # hashtags
    text = re.sub(r"@\S+", " ", text)               # raw @mentions
    text = re.sub(r"<redacted_mention>", " ", text)  # anonymized mentions
    text = re.sub(r"\n", "", text)                 # remove linebreaks
    text = re.sub(r" +", ' ', text)                   # remove repeated spaces
    return text

df_FFF["clean_text"] = df_FFF["text"].apply(strip_hashtags_mentions)
df_FFF["clean_word_count"] = df_FFF["clean_text"].str.split().str.len().fillna(0).astype(int)

n_empty = (df_FFF["clean_word_count"] == 0).sum()
print(f"posts with 0 words after stripping hashtags/mentions: {n_empty} "
      f"({n_empty / len(df_FFF) * 100:.1f}%)")

for threshold in (10, 20, 30, 40, 50):
    n = (df_FFF["clean_word_count"] >= threshold).sum()
    print(f"posts with >= {threshold} words remaining: {n} ({n / len(df_FFF) * 100:.1f}%)")

**Data-quality filter**: drop the posts with 0 words after stripping hashtags/mentions -- there is no caption left for the LLM to score, so these rows carry no framing information (as opposed to the 1-9 word posts, which are genuine short mobilization calls -- e.g. "Freitag = Streiktag!" -- and are deliberately kept). Applied here, upstream of every downstream notebook and script, so `data/processed/FFF_german.csv` and the LLM scores are consistent with each other.

In [ ]:
n_before = len(df_FFF)
df_FFF = df_FFF[df_FFF["clean_word_count"] > 0].reset_index(drop=True)
print(f"dropped {n_before - len(df_FFF)} posts with 0 words -- {len(df_FFF)} posts "
      f"remain for topic modeling, hashtag categorization, and the saved corpus")

In [ ]:
fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.histplot(df_FFF["clean_word_count"], bins=60, ax=ax)
for threshold, style in zip((10, 20, 30), ("--", ":", "--")):
    ax.axvline(threshold, color="black", linestyle=style, linewidth=1, label=f"{threshold} words")

ax.set_xlim(0, 100)
ax.set_title("Word Count per Post (Hashtags & Mentions Stripped)")
ax.set_xlabel("Words")
ax.set_ylabel("Number of Posts")
ax.legend()
#plot_style.savefig(fig, "FFF/clean_word_count_distribution")
plt.show()

In [ ]:
list(df_FFF[df_FFF["clean_word_count"] < 5]["clean_text"])

### Protest Logistics Heuristics

Quick glance: how many of the posts are dominated by protest logistics (date, time, place) and contain maybe no additional info/frames?



In [ ]:

# regex-based logistics markers: clock times, dates, and common German call-to-action phrases
TIME_RE = r"\b\d{1,2}[:.]\d{2}\s*uhr\b|\b\d{1,2}\s*uhr\b"
DATE_RE = r"\b\d{1,2}\.\d{1,2}\.(\d{2,4})?\b"
LOGISTICS_KEYWORDS = [
    "treffpunkt", "kundgebung", "startpunkt", "auftaktkundgebung",
    "wann:", "wo:", "ort:", "route:", "start:",
    "kommt alle", "kommt vorbei", "seid dabei", "sei dabei", "mobilisiert",
]

def has_logistics(text):
    t = str(text).lower()
    if re.search(TIME_RE, t) or re.search(DATE_RE, t):
        return True
    return any(kw in t for kw in LOGISTICS_KEYWORDS)

df_FFF["has_logistics"] = df_FFF["text"].apply(has_logistics)

n_logistics = df_FFF["has_logistics"].sum()
print(f"posts with logistics markers (time/date/call-to-action phrasing): "
      f"{n_logistics} ({n_logistics / len(df_FFF) * 100:.1f}%)")
print()
print("median clean word count, with vs. without logistics markers:")
print(df_FFF.groupby("has_logistics")["clean_word_count"].median())
print()
print("mean clean word count, with vs. without logistics markers:")
print(df_FFF.groupby("has_logistics")["clean_word_count"].mean().round(1))


In [ ]:
fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.boxplot(
    data=df_FFF, x="has_logistics", y="clean_word_count", ax=ax,
    showfliers=False,
)
ax.set_xticks([0, 1], ["No logistics markers", "Has logistics markers"])
ax.set_title("Word Count by Presence of Event-Logistics Markers")
ax.set_xlabel("")
ax.set_ylabel("Words (hashtags/mentions stripped)")
#plot_style.savefig(fig, "FFF/word_count_by_logistics")
plt.show()

Posts containing protest logistics actually are longer/contain more text, than those which do not have protest logistics. Maybe this is to be extected as they need to make a call to action and motivate people.

### Hashtag Co-occurrence

Pearson correlation between hashtag indicator variables (binary: present/absent per post) across the top 25 hashtags used by official FFF accounts: which sub-themes cluster together?

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

TOP_N_GENERIC = 30
MAX_DOC_FREQ = 0.5
EXTRA_TAGS = [
    "",
]

fff_hashtag_lists = (
    df_FFF["hashtags"].dropna().apply(json.loads).apply(lambda tags: [t.lower() for t in tags])
)
n_posts = len(fff_hashtag_lists)
tag_counts = Counter(tag for tags in fff_hashtag_lists for tag in tags)

generic_top_tags = [tag for tag, c in tag_counts.most_common() if c / n_posts <= MAX_DOC_FREQ][:TOP_N_GENERIC]
extra_present = [tag for tag in EXTRA_TAGS if tag in tag_counts]
top_n_tags = list(dict.fromkeys(generic_top_tags + extra_present))

hashtag_indicator = pd.DataFrame(
    {tag: fff_hashtag_lists.apply(lambda tags: tag in tags) for tag in top_n_tags}
)
hashtag_corr = hashtag_indicator.corr()


# cluster on the correlation structure (diagonal = 1 for a valid distance metric)
corr_for_clustering = hashtag_corr.fillna(1.0).to_numpy(copy=True)
np.fill_diagonal(corr_for_clustering, 1.0)
dist = 1 - corr_for_clustering
condensed_dist = squareform(dist, checks=False)
link = linkage(condensed_dist, method="average")
order = leaves_list(link)
ordered_tags = hashtag_corr.index[order]

hashtag_corr_sorted = hashtag_corr.loc[ordered_tags, ordered_tags]
for i in range(len(hashtag_corr_sorted)):
    hashtag_corr_sorted.iloc[i, i] = np.nan


fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    hashtag_corr_sorted, cmap=plot_style.PALETTE_SEQUENTIAL, center=0, square=True,
    linewidths=0.4, cbar_kws={"label": "Correlation (co-occurrence)"}, ax=ax,
)
ax.set_title("Hashtag Co-occurrence (Official FFF Accounts)")
#plot_style.savefig(fig, "FFF/hashtag_cooccurrence_heatmap")
plt.show()

### Hashtag Frequency Table (for Manual Categorization)

Compile a table of every hashtag used by official FFF accounts, sorted by frequency $\to$ will be used for manual topic detection based on hashtags with > 10 occurances.

In [ ]:
hashtag_counts_df = pd.DataFrame(tag_counts.most_common(), columns=["hashtag", "count"])

output_path = Path("../data/processed/fff_hashtag_counts.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
hashtag_counts_df.to_csv(output_path, index=False)

print(f"saved {len(hashtag_counts_df)} hashtags to {output_path}")
hashtag_counts_df.head(10)

#### Import compiled list

In [ ]:
hashtag_cat = pd.read_excel("../data/processed/fff_hashtag_categories.xlsx", header=0)
#hashtag_cat = hashtag_cat.rename(columns={0 : "concept", 1 : "hashtags"})
hashtag_cat

### Export: Axis E Candidate Labels (concept + examples, for LLM topic classification)

`hashtag_cat`'s `examples` column -- short illustrative terms per concept, not the full raw hashtag list -- is what `scripts/score_topic_llm_classify.py` shows the LLM as candidate labels (it asks for the 3 best-fitting labels per post, ranked, rather than a single pick without any grounding). `FFF` is excluded (branding, not a topic, same reasoning as `hashtag_topics` below); `unclassified` stays in as the catch-all, with an empty examples list.

In [ ]:
EXCLUDE_FROM_AXIS_E = {"FFF"}

axis_e_topics = {}
for _, row in hashtag_cat.iterrows():
    concept = str(row["concept"]).strip()
    if concept in EXCLUDE_FROM_AXIS_E:
        continue
    examples_raw = row["examples"]
    examples = [] if pd.isna(examples_raw) else [e.strip() for e in str(examples_raw).split(",") if e.strip()]
    axis_e_topics[concept] = examples

output_path = Path("../concepts/axis_e_topics_de.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(axis_e_topics, f, ensure_ascii=False, indent=2)

print(f"saved {len(axis_e_topics)} candidate topic labels to {output_path}")
axis_e_topics


### Applying the Manual Hashtag Categories

`hashtag_cat` above is wide -- one row per category, hashtags for that category comma-joined in one cell. Reshape it to a long `hashtag -> category` lookup, sanity-check it against the actual corpus (any hashtag assigned twice? any frequent corpus hashtag left uncategorized?), then attach categories to every post. This is the manual half of Axis E (topic/subject) in Project_Proposal.md §6.3.1 -- BERTopic supplies the other half in §5) below, and the two get cross-checked once topics are fit.

In [ ]:
hashtag_cat_long = hashtag_cat.copy()
hashtag_cat_long["hashtags"] = hashtag_cat_long["hashtags"].str.split(",")
hashtag_cat_long = hashtag_cat_long.explode("hashtags")
hashtag_cat_long["hashtags"] = hashtag_cat_long["hashtags"].str.strip().str.lower()
hashtag_cat_long = hashtag_cat_long.rename(columns={"hashtags": "hashtag"})

print(f"{len(hashtag_cat_long)} (category, hashtag) pairs across {hashtag_cat_long['concept'].nunique()} categories")

dupes = hashtag_cat_long[hashtag_cat_long.duplicated("hashtag", keep=False)].sort_values("hashtag")
if len(dupes):
    print(f"WARNING: {dupes['hashtag'].nunique()} hashtags assigned to more than one category -- "
          f"resolve by hand before treating this as a clean single-label mapping:")
    display(dupes)

hashtag_to_category = dict(zip(hashtag_cat_long["hashtag"], hashtag_cat_long["concept"]))


In [ ]:
# Which real, frequently-used hashtags in the corpus have no manual category at all?
uncategorized = [tag for tag, _ in tag_counts.most_common() if tag not in hashtag_to_category]
n_uncategorized_occurrences = sum(n for tag, n in tag_counts.most_common() if tag not in hashtag_to_category)
print(f"{len(uncategorized)}/{len(tag_counts)} distinct hashtags have no manual category "
      f"({n_uncategorized_occurrences} tag-post occurrences)")
print("most frequent uncategorized hashtags:")
pd.Series({t: tag_counts[t] for t in uncategorized[:20]})


### Data Cleaning before Topic Modeling

If topic modeling is applied on the full posts with hashtags and mentions, etc, these will dominate and bias the topics, BERT can find.

Hence, decided to strip all hashtags prior to topic modelling $\to$ BERT topics only based on posts text description.

In [ ]:

def categories_for(tags):
    if not isinstance(tags, list):
        return []
    return sorted({hashtag_to_category[t] for t in tags if t in hashtag_to_category})

df_FFF["hashtag_categories"] = fff_hashtag_lists.reindex(df_FFF.index).apply(categories_for)
df_FFF["hashtag_topics"] = df_FFF["hashtag_categories"].apply(
    lambda cats: [c for c in cats if c not in ("FFF", "unclassified")]
)

n_no_topic = (df_FFF["hashtag_topics"].str.len() == 0).sum()
print(f"posts with >=1 substantive manual category: {len(df_FFF) - n_no_topic} "
      f"({(len(df_FFF) - n_no_topic) / len(df_FFF) * 100:.1f}%)")
print(f"posts with none (no hashtags, or only FFF-branding/unclassified tags): {n_no_topic} "
      f"({n_no_topic / len(df_FFF) * 100:.1f}%)")


In [ ]:
category_counts = pd.Series(Counter(cat for cats in df_FFF["hashtag_topics"] for cat in cats)).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(x=category_counts.values, y=category_counts.index, order=category_counts.index, ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Posts per Manual Hashtag Category (multi-label)")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Category")
#plot_style.savefig(fig, "FFF/hashtag_category_counts")
plt.show()


In [ ]:
hashtag_cat_long[["concept", "hashtag"]].to_csv("../data/processed/fff_hashtag_categories_long.csv", index=False)
df_FFF[["id", "hashtag_categories", "hashtag_topics"]].to_csv(
    "../data/processed/fff_hashtag_topic_assignment.csv", index=False
)
print("saved long-format category lookup and per-post hashtag-topic assignment")


## 4) Manual Sample Per Crisis Window (Inductive Coding)

Manually check a sample of 5-10 posts per crisis window for topics and categories, relevant for classification.
Also, check presence of DPM frames, climate change vs. climate justice and indequality arenas.

$\to$ found the DPM frames, specially diagnosis/blame attribution and motivational calls to action. Climate change and justice are mentioned.
Inequality arenas are less obvious/prominent, but present, especially generational and global north vs. south.

$\to$ notes taken to sheet 2 in `fff_hashtag_categories.xlsx`

In [ ]:
N_SAMPLES_PER_WINDOW = 8
MIN_WORDS = 20
RANDOM_STATE = 42

resolved_windows = resolve_windows(CRISIS_WINDOWS, df_FFF["creation_time"].max())

for name, start, end, description in resolved_windows:
    window_df = filter_by_window(df_FFF, (name, start, end, description))
    eligible = window_df[window_df["clean_word_count"] > MIN_WORDS]
    n = min(N_SAMPLES_PER_WINDOW, len(eligible))
    sample = eligible.sample(n, random_state=RANDOM_STATE)

    print("=" * 78)
    print(f"WINDOW: {name}  ({start.date()} to {end.date()})")
    print(f"{description}")
    print(f"{len(window_df)} posts in window, {len(eligible)} with > {MIN_WORDS} words, showing {n}")
    print("=" * 78)
    for _, row in sample.iterrows():
        print(f"\n[{row['creation_time'].date()}] @{row['post_owner.username']}")
        print(row["text"])
        print("-" * 40)
    print()

## 5) Topic Modelling

**Diagnosis of the (removed) first run** (2 topics found, 99% of docs in one cluster labeled `fridaysforfuture_klimastreik_klimaschutz_straße`): since posts were drawn by the presence of `fridaysforfuture` in the post text (comprising text and hashtags), this mega-topic biased the topic modelling. Simply removing hashtag signs `#` did not solve the issue: the term still dominated the topic modelling.

$\to$ **Decision:** remove all hashtags, mentions etc. prior to topic modelling.

Furthermore:
1. **Smaller, explicit HDBSCAN** (`min_cluster_size=10`, `min_samples=5`) -- the themes you're after (democracy/voting, feminism) are genuinely small in this corpus (e.g. `#demokratie` 66 posts, `#feminismus` 2 posts) and would never survive the previous, more conservative default.
2. **Explicit UMAP with a fixed `random_state`** (`n_neighbors=10`) -- smaller neighborhood size preserves more local structure instead of smoothing everything into one blob, and now results are reproducible across reruns (the previous fit had no seed at all).
3. **`seed_topic_list`** nudging BERTopic toward your named themes (climate substance, protest/mobilization logistics, feminism, democracy/voting) -- semi-supervised guidance is a reasonable choice here given you already have strong theoretical priors about what should be there.
4. **Upgraded embedding model** to `paraphrase-multilingual-mpnet-base-v2` (larger/more accurate than the MiniLM model used when this had to run CPU-only) -- cheap now that you have real GPU headroom.
5. **`max_df=0.4` and bigrams** in the vectorizer -- keeps generic boilerplate out of topic *labels* too, and lets 2-word phrases (e.g. "klimastreik hamburg") show up as more specific labels than single words alone.


In [ ]:
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

In [ ]:
def clean_for_topic_model(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"<redacted_mention>", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#\S+", " ", text)  # drop hashtags entirely -- cluster on prose, not the boilerplate tag block
    return re.sub(r"\s+", " ", text).strip()

topic_docs = df_FFF["text"].apply(clean_for_topic_model)
non_empty = topic_docs.str.len() > 0
print(f"{(~non_empty).sum()} empty docs excluded from topic modeling")
topic_docs = topic_docs[non_empty].reset_index(drop=True)
topic_dates = df_FFF.loc[non_empty, "creation_time"].reset_index(drop=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device=device)

# Computed once, explicitly, so the refined re-fit below (after reduce_topics()) can reuse the
# same embeddings instead of re-running the transformer -- only the clustering step changes there.
topic_embeddings = embedding_model.encode(topic_docs.tolist(), show_progress_bar=True)

vectorizer_model = CountVectorizer(
    stop_words=list(DE_STOP_WORDS), min_df=1, max_df=1.0, ngram_range=(1, 2),
)

# min_cluster_size is the primary lever for topic count -- raised from 10 to ~1% of the corpus.
# n_neighbors back to UMAP's own default (was lowered to 10 to fight the old collapse-to-1-topic
# problem, which is now fixed upstream by stripping hashtags before embedding).
umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5, metric="euclidean",
    cluster_selection_method="leaf", prediction_data=True, # cluster_selection_method="eom"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(topic_docs.tolist(), embeddings=topic_embeddings)


In [ ]:
topic_info = topic_model.get_topic_info()

n_outliers = (pd.Series(topics) == -1).sum()
print(f"outlier topic (-1): {n_outliers} / {len(topics)} docs ({n_outliers / len(topics) * 100:.1f}%)")
print(f"{(topic_info['Topic'] != -1).sum()} coherent topics found")
topic_info.head(30)

**Diagnosis of 110 topics / 43.6% outliers**: `min_cluster_size=10`, `min_samples=5`, and `n_neighbors=10` were deliberately aggressive to escape the *previous* problem (everything collapsing into one cluster). That is now fixed structurally (hashtags stripped before embedding), so these same settings just over-fragment: $\to$ too many, tiny topics

**`min_cluster_size` is the primary lever for topic count** -- raise it and the topic count drops roughly monotonically, since HDBSCAN requires that many points before a cluster counts as a topic at all. A common starting heuristic is ~1% of the corpus (~40 here for n=4,104).



In [ ]:
# Cheap, interactive: merge the already-fitted 110 topics down to a target count.
# No re-embedding / no GPU needed for this cell -- just retry with different TARGET_N_TOPICS values.

TARGET_N_TOPICS = 30

topic_model.reduce_topics(topic_docs.tolist(), nr_topics=TARGET_N_TOPICS)

reduced_info = topic_model.get_topic_info()
n_outliers_reduced = (pd.Series(topic_model.topics_) == -1).sum()
print(f"outlier topic (-1): {n_outliers_reduced} / {len(topic_model.topics_)} docs "
      f"({n_outliers_reduced / len(topic_model.topics_) * 100:.1f}%)")
print(f"{(reduced_info['Topic'] != -1).sum()} topics after reduction (target was {TARGET_N_TOPICS})")
reduced_info.head(30)

# 29 topics emerge as the max, even when setting a higher target

### Refined Re-fit at the Chosen Topic Count

`reduce_topics()` above is only exploratory: merges the fragmented 110-topic clustering by representation similarity

**Now:** re-fit BERT topic modelling to aim for around 30-40 topics 


In [ ]:

REFINED_MIN_CLUSTER_SIZE = max(10, len(topic_docs) // 100)  

refined_umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42,
)
refined_hdbscan_model = HDBSCAN(
    min_cluster_size=REFINED_MIN_CLUSTER_SIZE, min_samples=5, metric="euclidean",
    cluster_selection_method="leaf", prediction_data=True,
)

refined_topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=refined_umap_model,
    hdbscan_model=refined_hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)
refined_topics, _ = refined_topic_model.fit_transform(topic_docs.tolist(), embeddings=topic_embeddings)

refined_topic_info = refined_topic_model.get_topic_info()
n_outliers_refined = (pd.Series(refined_topics) == -1).sum()
print(f"min_cluster_size={REFINED_MIN_CLUSTER_SIZE} -- outlier topic (-1): {n_outliers_refined} / "
      f"{len(refined_topics)} docs ({n_outliers_refined / len(refined_topics) * 100:.1f}%)")
print(f"{(refined_topic_info['Topic'] != -1).sum()} topics from the refined re-fit (target was {TARGET_N_TOPICS})")
refined_topic_info.head(30)


In [ ]:
# Set the refined topic model as the one to work with from here on!

topic_model = refined_topic_model
topics = refined_topics
topic_info = refined_topic_info

# and export it
topic_info.to_csv("../data/processed/fff_topics.csv", index = False)

In [ ]:
top_topics = topic_info[topic_info["Topic"] != -1].head(15)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(data=top_topics, y="Name", x="Count", order=top_topics["Name"], ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Top 15 BERTopic Topics by Size")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Topic")
#plot_style.savefig(fig, "FFF/bertopic_top_topics")
plt.show()

In [ ]:
topics_over_time = topic_model.topics_over_time(topic_docs.tolist(), topic_dates.tolist(), nr_bins=40)

topic_names = topic_info.set_index("Topic")["Name"]
top_topic_ids = top_topics["Topic"].head(8).tolist()
plot_data = topics_over_time[topics_over_time["Topic"].isin(top_topic_ids)].copy()
plot_data["Topic Name"] = plot_data["Topic"].map(topic_names)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.lineplot(data=plot_data, x="Timestamp", y="Frequency", hue="Topic Name", ax=ax)
ax.set_title("Top Topics Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Posts per Time Bin")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
#plot_style.savefig(fig, "FFF/bertopic_topics_over_time")
plt.show()

In [ ]:
df_FFF["topic"] = pd.NA
df_FFF.loc[non_empty, "topic"] = topics
df_FFF["topic_name"] = df_FFF["topic"].map(topic_names)

df_FFF["topic_name"].value_counts(dropna=False)

### Cross-Check: Manual Categories vs. BERTopic Topics

Axis E (§6.3.1) combines BERTopic's discovered topics with this manual hashtag codebook. For each BERTopic topic, the most common manual category among its posts -- a quick read on whether the two agree, and where they diverge.

In [ ]:
df_topic_valid = df_FFF.dropna(subset=["topic"])
df_topic_valid = df_topic_valid[df_topic_valid["topic"] != -1]

topic_sizes = df_topic_valid.groupby("topic_name").size().rename("n_posts")

# coverage: share of this topic's posts that have >=1 substantive manual category at all.
coverage = (
    df_topic_valid.assign(has_category=df_topic_valid["hashtag_topics"].str.len() > 0)
    .groupby("topic_name")["has_category"]
    .mean()
    .rename("coverage")
)

# share: among the posts that DO have a category, how concentrated on the single most common
# one. value_counts() drops empty hashtag_topics before normalizing (explode() turns [] into
# NaN), so this is NOT the same measurement as coverage -- a topic can show a high share while
# being almost entirely uncovered, if the few tagged posts happen to agree. Read both together.
top_category = (
    df_topic_valid.explode("hashtag_topics")
    .groupby("topic_name")["hashtag_topics"]
    .value_counts(normalize=True)
    .groupby(level=0)
    .head(1)
    .rename("share")
    .reset_index()
    .rename(columns={"hashtag_topics": "top_category"})
    .set_index("topic_name")
)

topic_vs_category = (
    topic_sizes.to_frame()
    .join(coverage)
    .join(top_category)
    .sort_values(["coverage", "share"])
)
topic_vs_category


In [ ]:
fully_uncovered = topic_vs_category[topic_vs_category["coverage"] == 0]
print(f"{len(fully_uncovered)}/{len(topic_vs_category)} reduced topics have ZERO posts "
      f"with any manual hashtag category -- content BERTopic found that the hashtag "
      f"codebook doesn't represent at all.")
fully_uncovered


### Topics Flagged for Manual Review

Below-threshold on *both* coverage and share -- either genuinely uncovered by hashtags, or covered but too inconsistently tagged for a single category to represent them. `fully_uncovered` above is the extreme case (already included here, since `coverage == 0` also fails the coverage threshold). Thresholds are a starting point, not a fixed rule -- tune them once you see how many topics they flag.

In [ ]:
COVERAGE_THRESHOLD = 0.5
SHARE_THRESHOLD = 0.5

needs_review = topic_vs_category[
    (topic_vs_category["coverage"] < COVERAGE_THRESHOLD)
    & ((topic_vs_category["share"] < SHARE_THRESHOLD) | topic_vs_category["share"].isna())
].sort_values(["coverage", "share"])

print(f"{len(needs_review)}/{len(topic_vs_category)} topics flagged (coverage < {COVERAGE_THRESHOLD:.0%} "
      f"AND top-category share < {SHARE_THRESHOLD:.0%} among covered posts, treating no-coverage-at-all "
      f"topics as failing the share check too)")
needs_review


### Representative Posts for Flagged Topics

Text + raw hashtags + assigned manual categories side by side, so you can actually judge each flagged topic instead of reading the aggregate numbers alone.

In [ ]:
N_EXAMPLES_PER_TOPIC = 5
RANDOM_STATE = 42

review_cols = ["id", "text", "hashtags", "hashtag_categories", "hashtag_topics"]
examples = []
for topic_name in needs_review.index:
    topic_posts = df_topic_valid[df_topic_valid["topic_name"] == topic_name]
    sample = topic_posts.sample(min(N_EXAMPLES_PER_TOPIC, len(topic_posts)), random_state=RANDOM_STATE)
    examples.append(sample[review_cols].assign(topic_name=topic_name))

review_examples = (
    pd.concat(examples, ignore_index=True) if examples else pd.DataFrame(columns=review_cols + ["topic_name"])
)
review_examples.to_csv("../data/processed/fff_topics_needing_review.csv", index=False)
print(f"saved {len(review_examples)} example posts across {len(needs_review)} flagged topics for manual reading")
review_examples.head(20)


### Topic-Naming Worksheet (prep for item 4: LLM topic-scoring candidate list)

The consolidated candidate label set for LLM topic-scoring = your manual hashtag categories + whatever BERTopic found that they don't already cover. The flagged topics above are exactly the ones that need a genuinely NEW short label from you -- BERTopic's own `Name` (top keywords, e.g. `12_11_18_00_lützerath`) isn't usable as an LLM answer option as-is. Fill in `extended_label` by hand where you want a new candidate; leave it blank to just reuse `top_manual_category` for topics that are already well explained by your existing codebook -- same "fill in by hand" pattern as `fff_hashtag_categories.xlsx` itself.

In [ ]:
topic_worksheet = (
    topic_vs_category
    .join(reduced_info.set_index("Name")[["Topic", "Representative_Docs"]])
    .reset_index()
    .rename(columns={"top_category": "top_manual_category"})
    .sort_values(["coverage", "share"])
)
topic_worksheet["extended_label"] = ""  # fill in by hand, see markdown above

output_path = "../data/processed/fff_topic_naming_worksheet.csv"
topic_worksheet.to_csv(output_path, index=False)
print(f"saved {len(topic_worksheet)} reduced topics to {output_path} -- "
      f"fill in extended_label for the flagged (low coverage/share) rows")
topic_worksheet.head(20)


### Next Step for Item 4: LLM Topic-Scoring Candidate List

Once `fff_topic_naming_worksheet.csv` is filled in, the consolidated candidate list = manual hashtag categories + the `extended_label` values you added for genuinely uncovered topics.

One thing to settle before wiring up the actual scoring run: `score_axis_llm_ddr.py` is built around the `ddr` package's per-concept **Likert rating** -- each concept gets its own independent 0-4 intensity score. That's a good fit for DPM's 3 concepts, but asking for a 0-4 score on *each* of ~30-40 topic labels per post is a much heavier, more failure-prone prompt than a **classification** task ("which of these labels apply to this post"). I'd build a lighter, dedicated classification prompt/script for this axis rather than force it through the existing DDR Likert-per-concept scorer -- flag once the worksheet is filled in and I'll set that up as a separate script (same checkpointing/resume pattern, different prompt shape).

## Save Enriched Corpus

Writes `clean_text`, `clean_word_count`, `has_logistics`, the hashtag categories and the
BERTopic topic back into `data/processed/FFF_german.csv`, which every downstream notebook
and `scripts/build_dashboard.py` read.

In [ ]:
df_FFF.to_csv("../data/processed/FFF_german.csv", index=False)
print(f"saved {len(df_FFF)} posts with {df_FFF.shape[1]} columns to ../data/processed/FFF_german.csv")

## Takeaway

**Text length & logistics heuristics**: your fear that posts are mostly thin protest-call logistics doesn't hold up. 30.5% of posts contain an explicit event-logistics marker (clock time, date, or "Treffpunkt"/"kommt alle"-style phrase), but those posts are actually **longer**, not shorter (median 59 words vs. 45 for the rest). 94.1% of posts have ≥10 words and 83.9% have ≥20 words after stripping hashtags/mentions — DPM framing coding looks viable on text length grounds alone.

**Hashtag co-occurrence**: confirms real, distinguishable sub-themes beyond generic climate branding — a democracy/anti-AfD cluster (`#demokratie` ↔ `#fckafd`) and a feminism cluster (`#feminism` ↔ `#intersectionalfeminism`) both stand out clearly once the near-universal `#fridaysforfuture` tag (82.3% of posts, essentially zero variance) is excluded.

**Topic modeling**: the first BERTopic run collapsed to one giant cluster because hashtag boilerplate dominated the embeddings (see §4's diagnosis). Revised with hashtag-free embedding text, smaller/explicit HDBSCAN clustering, and a seed topic list targeting climate/mobilization/feminism/democracy — rerun on the GPU server to see whether it now recovers the sub-themes the hashtag correlation analysis already points to.

The zero-shot/LLM-based DPM scoring step (§6.3) is deferred to a later stage per your steer.